# RC9.2.2 RC6 — production schedule on Colab
## Variant A: no Google Drive

Release `L6.3.2.8-RC9.2.2-PRODUCTION-HARDENED-RC6`. Full instructions:
`PRODUCTION_RUN_GUIDE.md` in the package root.

**What this notebook does**
1. Installs the pinned solver (OR-Tools 9.15.6755) and scipy.
2. Asks you to upload the package ZIP.
3. Runs your workbook (`MY_WORKBOOK`) or the packaged scenarios. Before any
   solving, the runner checks the engine's sha256, the runtime, and runs the
   full offline test gate (15-30 min on a 2-CPU Colab). If anything fails it
   refuses to run.
4. Every schedule it publishes has passed the independent validator with 0 hard
   failures. Exit code 0 = approved schedule; anything else = not approved.

**Recommended settings (measured):** `MODE = QUICK`, `SEEDS = 0` (two 1-hour
seeds, best kept). About 1 h on 4+ CPUs, 2 h on 2 CPUs, plus the gate.
`MODE = DEEP` = best of 4 x 1-hour seeds.

**Do not use for production yet:** `SINGLE_LONG_RUN = True` (the one-long-run
DEEP path is still being verified end to end), and `SKIP_GUARDS = True`.

**Runtime type:** CPU. A GPU does nothing for this solver. Pick High-RAM if offered.

Everything stays in the Colab VM: **if the VM disconnects, the results are
lost.** For long runs use the Drive notebook (Variant B).

In [ ]:
#@title 1. Environment — pin the solver version { display-mode: "form" }
# ortools 9.15.6755 is the pinned RC9.2.2 solver runtime (the runner refuses any other).
# A different solver build can change candidate selection, so pin it.
!pip -q install "ortools==9.15.6755" "openpyxl>=3.1" "scipy>=1.11" 2>&1 | tail -2

import multiprocessing, platform, sys
print("python :", platform.python_version())
print("cpus   :", multiprocessing.cpu_count())
import ortools; print("ortools:", ortools.__version__)
import scipy; print("scipy  :", scipy.__version__)

In [ ]:
#@title 2. Upload the package ZIP { display-mode: "form" }
from google.colab import files
import pathlib, zipfile, io, os

os.makedirs("/content/rc922", exist_ok=True)
uploaded = files.upload()          # choose RC9_2_2_PRODUCTION_PACKAGE.zip (the RC5 line; RC5 itself shipped as RC9_2_2_FIX_VALIDATION_RC5.zip)
name = next(iter(uploaded))
with zipfile.ZipFile(io.BytesIO(uploaded[name])) as z:
    z.extractall("/content/rc922")
print("extracted to /content/rc922")
!ls /content/rc922

In [ ]:
#@title 3. Locate the package { display-mode: "form" }
import pathlib
cands = list(pathlib.Path("/content/rc922").rglob("SCENARIOS.json"))
assert cands, "SCENARIOS.json not found - did the ZIP extract correctly?"
PACKAGE_ROOT = str(cands[0].parent)
RESULTS_ROOT = pathlib.Path(PACKAGE_ROOT) / "results"
RESULTS_ROOT.mkdir(exist_ok=True)
print("PACKAGE_ROOT =", PACKAGE_ROOT)
print("RESULTS_ROOT =", RESULTS_ROOT)

In [ ]:
#@title 4. Verify the package before running anything { display-mode: "form" }
# Shows what the package contains and which engine it will run. The full
# safety gate (engine sha256, runtime, all offline tests) runs automatically at
# the start of the next step and refuses to run if anything fails.
import subprocess, sys, json, pathlib
ROOT = pathlib.Path(PACKAGE_ROOT)

manifest = json.loads((ROOT / "SCENARIOS.json").read_text())
print("package:", manifest["package"])
print("engine :", manifest["engine_release"])
print("sha256 :", manifest["engine_sha256"][:32], "…")
print()
for row in manifest["scenarios"]:
    print(f"  {row['scenario_id']:14} {row['input']}")
print()

proc = subprocess.run([sys.executable, str(ROOT / "runners" / "rc922_runner.py"),
                       "--package-root", str(ROOT), "--gate-only",
                       "--results-root", str(ROOT / "results")],
                      capture_output=True, text=True)
print("(no results yet is expected on a fresh package)")

In [ ]:
#@title 4b. Check your workbook first (seconds) { display-mode: "form" }
WORKBOOK_TO_CHECK = ""   #@param {type:"string"}
# Full path of the .xlsx you are about to schedule. This runs the same input
# checks the production run starts with, in seconds, so a bad cell is found now
# rather than after the 15-30 min safety gate. ACCEPTED = ready to run.
import subprocess, sys
if WORKBOOK_TO_CHECK.strip():
    proc = subprocess.run([sys.executable, str(ROOT / "tools" / "check_input_workbook.py"),
                           WORKBOOK_TO_CHECK.strip()], capture_output=True, text=True)
    print(proc.stdout)
    if proc.returncode:
        print(proc.stderr[-2000:])
        print("Fix the items above in the workbook, upload it again, and re-run this cell.")
else:
    print("Set WORKBOOK_TO_CHECK to the path of your workbook.")

In [ ]:
#@title 5. Run { display-mode: "form" }
MY_WORKBOOK = ""   #@param {type:"string"}
# MY_WORKBOOK: full path of your .xlsx (e.g. /content/drive/MyDrive/schedules/week42.xlsx,
# or /content/week42.xlsx after uploading in the Files panel). Leave empty to run
# the packaged scenarios instead. Results go to RESULTS_ROOT/<WORKBOOK NAME>/.
MODE    = "QUICK"  #@param ["SMOKE", "QUICK", "DEEP", "OVERNIGHT"]
SEEDS   = 0        #@param {type:"integer"}
# SEEDS = 0 is automatic (measured, evidence/seed_portfolio_ab/RESULT.txt):
#   QUICK -> best of 2 x 1 h seeds, DEEP -> best of 4, OVERNIGHT -> best of 6.
# Seeds run side by side on 4+ CPUs and one after another on 2 CPUs.
STAGE   = "FULL_SCHEDULE"  #@param ["FULL_SCHEDULE", "BEFORE_BREAKS_ONLY"]
LANGUAGE_WORKING_WINDOW = "workbook" #@param ["workbook", "OFF", "MINIMUM_ROWS", "ALL_ROWS", "REQUIRED_LANGUAGE_ONLY"]
# "workbook" = use the value in your Instructions sheet (recommended).
SINGLE_LONG_RUN = False #@param {type:"boolean"}
# NOT for production yet: the one-long-run DEEP/OVERNIGHT path is still being
# verified end to end. Keep False.
ONLY    = ""       #@param {type:"string"}
SHARD   = 0        #@param {type:"integer"}
SHARDS  = 1        #@param {type:"integer"}
# ONLY / SHARD / SHARDS pick packaged scenarios; ignored when MY_WORKBOOK is set.
TIME_LIMIT = 0     #@param {type:"integer"}
# 0 = the mode's own budget (QUICK 3600 s per seed). MODE sets the budget AND the
# phase reserves, so change MODE rather than TIME_LIMIT.
RESUME = True       #@param {type:"boolean"}
OVERWRITE = False   #@param {type:"boolean"}
# RESUME continues an interrupted run from its checkpoints. OVERWRITE must be
# ticked to replace a workbook/scenario that already has results in RESULTS_ROOT.
SKIP_GUARDS = False #@param {type:"boolean"}
# Keep False: the guards are what make the output safe to use.

import subprocess, sys, multiprocessing, os, signal
cmd = [sys.executable, "-u", str(ROOT / "runners" / "rc922_runner.py"),
       "--package-root", str(ROOT),
       "--results-root", str(RESULTS_ROOT),
       "--shard", str(SHARD), "--shards", str(SHARDS),
       "--mode", MODE,
       "--stage", STAGE,
       "--num-workers", str(multiprocessing.cpu_count())]
AUTO_SEEDS = {"SMOKE": 1, "QUICK": 2, "DEEP": 0, "OVERNIGHT": 0}   # 0: runner default (DEEP 4, OVERNIGHT 6)
seeds = int(SEEDS) if int(SEEDS) > 0 else AUTO_SEEDS[MODE]
if seeds:                            cmd += ["--seeds", str(seeds)]
if SINGLE_LONG_RUN:                  cmd += ["--single-run"]
if TIME_LIMIT:                       cmd += ["--time-limit", str(TIME_LIMIT)]
if ONLY.strip():                     cmd += ["--only", ONLY.strip()]
if MY_WORKBOOK.strip():              cmd += ["--input", MY_WORKBOOK.strip()]
if LANGUAGE_WORKING_WINDOW != "workbook":
    cmd += ["--language-working-window", LANGUAGE_WORKING_WINDOW]
if SKIP_GUARDS:                      cmd += ["--skip-guards"]
if RESUME:                           cmd += ["--resume"]
if OVERWRITE:                        cmd += ["--overwrite"]

print(" ".join(cmd), "\n")
# The runner and everything it starts (seed runs, engines) live in their own
# process group, so interrupting this cell stops all of them, not just the top one.
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1, start_new_session=(os.name == "posix"))
try:
    for line in proc.stdout:
        print(line, end="")
    print("\nexit code:", proc.wait())
except KeyboardInterrupt:
    print("\nInterrupt received: asking the runner to stop safely and preserve checkpoints...")
    try:
        proc.send_signal(signal.SIGINT)
        print("runner exit code:", proc.wait(timeout=30))
    except subprocess.TimeoutExpired:
        if os.name == "posix":
            os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
        else:
            proc.terminate()
        proc.wait(timeout=30)
    raise


In [ ]:
#@title 6. Score the gates (re-runnable on its own) { display-mode: "form" }
import subprocess, sys
proc = subprocess.run([sys.executable, str(ROOT / "tools" / "release_gate_report.py"),
                       str(RESULTS_ROOT), "--out-dir", str(RESULTS_ROOT / "_gate_report")],
                      capture_output=True, text=True)
print(proc.stdout)
if proc.returncode: print(proc.stderr[-3000:])

## 7. Download the results

Send back the **whole** ZIP. It contains `RUN_LEDGER.json` (which scenarios ran,
on what hardware, wall clock vs budget) and `_gate_report/` — both are needed to
interpret the numbers.

In [ ]:
#@title 7. Package and download results { display-mode: "form" }
import shutil, time
from google.colab import files
stamp = time.strftime("%Y%m%d_%H%M%S")
out = f"/content/RC922_RESULTS_{stamp}"
shutil.make_archive(out, "zip", str(RESULTS_ROOT))
print("size:", round(os.path.getsize(out + ".zip")/1e6, 1), "MB")
files.download(out + ".zip")